___

# <font color= #99C8F5> **Modelo SARIMA: NBA API** </font>
#### <font color= #2E9AFE> `Modelos No Lineales Para Pronósticos`</font>
<Strong> Sarah Beltrán, Sofía Maldoando & Aissa Berenice </Strong>

_22/02/2026._

___

Para esta tarea, buscamos usar un modelo SARIMA para predecir la cantidad de puntos totales que se van a anotar en un día en la NBA. Vamos a hacer nuestras predicciones con datos de la temporada 2024-2025.

In [15]:
# Imports
# Data y Generales 
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import numpy as np

# Pruebas y Modelado
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error

# Gráficas y Visualización
import plotly.graph_objects as go
import nbformat
from plotly.subplots import make_subplots

In [2]:
games = leaguegamefinder.LeagueGameFinder(
    season_type_nullable='Regular Season'
).get_data_frames()[0]
df_games = pd.DataFrame(games)

# converts date
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])

# regular season 2024-2025
# SEASON_ID: '22024'
df_games = df_games[df_games['SEASON_ID'] == '22024']

# no play-in/offs
df_clean = df_games[df_games['GAME_DATE'] <= '2025-02-15'].copy()

# ended games
df_clean = df_clean[df_clean['WL'].notna()]

# total points per game
df_clean['total_points'] = df_clean['PTS']

# daily time series points per day
ts_nba = (
    df_clean
    .groupby('GAME_DATE')['total_points']
    .sum()
    .asfreq('D')
    .fillna(0)
)

In [3]:
# original time series graph
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ts_nba.index,
        y=ts_nba.values,
        mode='lines',
        name='Puntos Diarios'
    )
)

fig.update_layout(
    title='Volumen Diario de Puntos en la NBA (Regular Season)',
    xaxis_title='Fecha',
    yaxis_title='Total de Puntos'
)

fig.show()

In [4]:
# stationariry tests

def check_stationarity(series, title="series"):
    result = adfuller(series.dropna())
    print(f'ADF test: {title}')
    print(f'statistics ADF: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    is_stationary = result[1] < 0.05
    print(f"Is it stationary? {'YES' if is_stationary else 'NO'}\n")
    return is_stationary

# 1. original
check_stationarity(ts_nba, "original level (NBA)")

# 2. first diff
ts_nba_diff = ts_nba.diff()

# 3. diff series
check_stationarity(ts_nba_diff, "first differentiation (d=1)")

# comparative subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "original series (NBA – not stationary)",
        "differentiated series (NBA – stationary d=1)"
    )
)

# Serie original
fig.add_trace(
    go.Scatter(x=ts_nba.index, y=ts_nba, name='original'),
    row=1, col=1
)

# Serie diferenciada
fig.add_trace(
    go.Scatter(x=ts_nba_diff.index, y=ts_nba_diff, name='differentiated'),
    row=1, col=2
)

fig.update_layout(
    title_text="comparative differentiation effect (NBA)",
    showlegend=False,
    height=500
)

fig.show()


ADF test: original level (NBA)
statistics ADF: -2.3656
p-value: 0.1517
Is it stationary? NO

ADF test: first differentiation (d=1)
statistics ADF: -4.4481
p-value: 0.0002
Is it stationary? YES



In [5]:
# using diff series cause d=1
ts_analysis = ts_nba.diff().dropna()

lags = 30      # 30 days
alpha = 0.05  

acf_vals = acf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]
pacf_vals = pacf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]

n = len(ts_analysis)
conf_interval = 1.96 / np.sqrt(n)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "autocorrelation function (ACF) – determine MA(q)",
        "partial autocorrelation (PACF) – determine AR(p)"
    ),
    vertical_spacing=0.15
)

# ACF
fig.add_trace(
    go.Bar(
        x=list(range(1, lags + 1)),
        y=acf_vals,
        name='ACF',
        showlegend=False
    ),
    row=1, col=1
)

fig.add_shape(
    type="rect",
    x0=0.5, y0=-conf_interval, x1=lags + 0.5, y1=conf_interval,
    line=dict(width=0),
    fillcolor="rgba(0,0,0,0.1)",
    row=1, col=1
)

fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=1, col=1)

# PACF
fig.add_trace(
    go.Bar(
        x=list(range(1, lags + 1)),
        y=pacf_vals,
        name='PACF',
        showlegend=False
    ),
    row=2, col=1
)

fig.add_shape(
    type="rect",
    x0=0.5, y0=-conf_interval, x1=lags + 0.5, y1=conf_interval,
    line=dict(width=0),
    fillcolor="rgba(0,0,0,0.1)",
    row=2, col=1
)

fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=2, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=2, col=1)

fig.update_layout(
    title="<b>structure diagnosis: ACF y PACF (NBA)</b><br><sup>diff. series</sup>",
    template="plotly_white",
    height=700,
    bargap=0.8
)

# lags (red lines)
# for i in [7, 14, 21, 28]:
#     fig.add_vline(x=i, line_width=1, line_dash="dot", line_color="red", opacity=0.5)

fig.show()

La estacionalidad no se puede ver tan directamente en la serie de tiempo, ya que el calendario de la NBA es bastante más mezclado al de la NFL (que solo juega en fines de semana y jueves) y el de MLB (que juega toda la semana pero con días de viaje medio establecidos). Sin embargo, hay patrones generales en toda la semana que hacen sentido (por ejemplo, se juega más en fines de semana que al inicio de la misma). Esto nos llevó a probar usar un valor $s$ de 7, es decir, estacionalidad semanal. 

In [39]:
#Realizamos modelo y graficamos
TEST_DAYS = 14

train = ts_nba.iloc[:-TEST_DAYS]
test = ts_nba.iloc[-TEST_DAYS:]

model = SARIMAX(train,
                order=(1, 0, 1),
                seasonal_order=(1, 1, 1, 7))

results = model.fit(disp=False)

# Predecimos n pasos hacia el futuro (donde n = tamaño del test)
forecast_object = results.get_forecast(steps=len(test))
forecast_vals = forecast_object.predicted_mean
conf_int = forecast_object.conf_int(alpha=0.05) # Intervalo del 95%

# Metricas de error
rmse = np.sqrt(mean_squared_error(test, forecast_vals))
mape = mean_absolute_percentage_error(test, forecast_vals)
mae = mean_absolute_error(test, forecast_vals)


print(f"\n--- Errores del modelo ---")
print(f"RMSE: {rmse:.2f} Puntos")
print(f"MAPE: {mape:.2%}")
print(f"MAE: {mae:.2f} Puntos")

print(results.summary())

# Grafica
fig = go.Figure()

# Train
fig.add_trace(go.Scatter(
    x=train.index, y=train,
    mode='lines',
    name='Train',
    line=dict(color='rgba(100, 100, 100, 0.6)', width=1.5)
))

# Test
fig.add_trace(go.Scatter(
    x=test.index, y=test,
    name='Test',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=6)
))

# Forecast
fig.add_trace(go.Scatter(
    x=test.index, y=forecast_vals,
    name='SARIMA',
    line=dict(color='#ff7f0e', width=3, dash='dot')
))

# Intervalos de Confianza
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 0],
    mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
))
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 1],
    mode='lines', line=dict(width=0), fill='tonexty',
    fillcolor='rgba(255, 127, 14, 0.2)',
    name='Int. Confianza 95%', hoverinfo='skip'
))

# Titulos
fig.update_layout(
    title=f'<b>Modelo SARIMA: Pronóstico de Puntos en la NBA</b>',
    xaxis_title='Fecha',
    yaxis_title='Total de Puntos',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.8)'),
    hovermode="x unified"
)

fig.show()


--- Errores del modelo ---
RMSE: 556.01 Puntos
MAPE: 21.42%
MAE: 372.83 Puntos
                                     SARIMAX Results                                     
Dep. Variable:                      total_points   No. Observations:                  101
Model:             SARIMAX(1, 0, 1)x(1, 1, 1, 7)   Log Likelihood                -744.289
Date:                           Thu, 19 Feb 2026   AIC                           1498.578
Time:                                   10:46:33   BIC                           1511.294
Sample:                               10-22-2024   HQIC                          1503.715
                                    - 01-30-2025                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.7421      0.121     -6.15